# Regularization Techniques: Taming Overfitting

**What we'll learn:**
- Why neural networks overfit and how regularization helps
- **L2 regularization** (Ridge) - penalizing large weights
- **L1 regularization** (Lasso) - encouraging sparsity
- **Weight decay** - the optimizer's perspective on L2
- **Dropout** - randomly dropping neurons during training
- **Early stopping** - stopping before we overfit

**Why it matters:**

Without regularization, models memorize training data instead of learning patterns. Regularization forces models to learn simpler, more generalizable solutions.

**Intuition we'll build:**

Think of regularization like adding friction to a skateboard - it prevents you from going too fast and losing control, ensuring a smoother, more stable ride.

## 1. Setup

Let's import our tools and set up reproducibility.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

Set random seed for reproducibility and configure device.

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 2. The Overfitting Problem

Before diving into solutions, let's see the problem in action.

### 2.1 Create Synthetic Data

We'll generate a simple noisy sine wave with limited training samples. This makes overfitting easy to observe.

In [ ]:
def create_noisy_sine_data(n_samples=100, noise_level=0.1):
    """Generate noisy sine wave data for regression."""
    X = np.linspace(0, 2*np.pi, n_samples)
    y = np.sin(X) + np.random.randn(n_samples) * noise_level
    return X.reshape(-1, 1).astype(np.float32), y.astype(np.float32)

# Small training set (easy to overfit)
X_train, y_train = create_noisy_sine_data(n_samples=50, noise_level=0.15)

# Large test set for evaluation
X_test, y_test = create_noisy_sine_data(n_samples=200, noise_level=0.15)

# Convert to tensors
X_train_t = torch.from_numpy(X_train).to(device)
y_train_t = torch.from_numpy(y_train).to(device)
X_test_t = torch.from_numpy(X_test).to(device)
y_test_t = torch.from_numpy(y_test).to(device)

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Visualize the data to see what we're working with.

In [ ]:
plt.figure(figsize=(10, 4))
plt.scatter(X_train, y_train, alpha=0.6, label='Train', s=50)
plt.scatter(X_test, y_test, alpha=0.3, label='Test', s=20)
plt.plot(X_test, np.sin(X_test), 'r--', label='True function', linewidth=2)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Noisy Sine Wave Data')
plt.grid(True, alpha=0.3)
plt.show()

### 2.2 Build an Overparameterized Model

A model with **too many parameters** relative to the data will easily overfit. Let's create one with lots of capacity.

In [ ]:
class OverparameterizedNet(nn.Module):
    """A neural network with excessive capacity for the task."""
    def __init__(self, hidden_size=100):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

model = OverparameterizedNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model has {n_params:,} parameters for only {len(X_train)} training samples!")
print(f"Ratio: {n_params / len(X_train):.1f} parameters per training sample")

### 2.3 Train Without Regularization

Let's train this model without any regularization and watch it overfit.

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, epochs=1000, lr=0.01):
    """Train model and track train/test loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        pred_train = model(X_train).squeeze()
        loss_train = F.mse_loss(pred_train, y_train)
        loss_train.backward()
        optimizer.step()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            pred_test = model(X_test).squeeze()
            loss_test = F.mse_loss(pred_test, y_test)
        
        train_losses.append(loss_train.item())
        test_losses.append(loss_test.item())
        
        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}: Train Loss = {loss_train:.4f}, Test Loss = {loss_test:.4f}")
    
    return train_losses, test_losses

# Reset model
set_seed(42)
model_no_reg = OverparameterizedNet().to(device)

train_losses_no_reg, test_losses_no_reg = train_model(
    model_no_reg, X_train_t, y_train_t, X_test_t, y_test_t, epochs=1000
)

Visualize the classic overfitting pattern: train loss keeps decreasing while test loss increases.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses_no_reg, label='Train Loss')
plt.plot(test_losses_no_reg, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.title('Overfitting: Train vs Test Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
model_no_reg.eval()
with torch.no_grad():
    X_plot = torch.linspace(0, 2*np.pi, 300).reshape(-1, 1).to(device)
    y_pred = model_no_reg(X_plot).squeeze().cpu().numpy()

plt.scatter(X_train, y_train, alpha=0.6, label='Train Data', s=50)
plt.plot(X_plot.cpu().numpy(), y_pred, 'r-', label='Model Prediction', linewidth=2)
plt.plot(X_plot.cpu().numpy(), np.sin(X_plot.cpu().numpy()), 'g--', label='True Function', linewidth=2)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Overfit Model: Wiggly Predictions')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observation:** The model memorizes training points instead of learning the smooth sine wave. Notice the erratic wiggles between training points!

## 3. L2 Regularization (Ridge)

**Core idea:** Penalize large weights by adding $\frac{\lambda}{2} \sum w^2$ to the loss.

**Intuition:** Large weights allow the model to fit noise aggressively. By penalizing them, we encourage smoother, more generalizable functions.

### 3.1 Understanding L2 Penalty

The modified loss function becomes:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}} + \frac{\lambda}{2} \sum_{i} w_i^2$$

where $\lambda$ controls regularization strength.

In [ ]:
def compute_l2_penalty(model):
    """Compute L2 penalty (sum of squared weights)."""
    l2_penalty = 0.0
    for param in model.parameters():
        l2_penalty += torch.sum(param ** 2)
    return l2_penalty

# Check L2 penalty of our overfit model
l2_penalty = compute_l2_penalty(model_no_reg)
print(f"L2 penalty of overfit model: {l2_penalty:.2f}")
print(f"This is added to loss scaled by λ/2")

### 3.2 Train with L2 Regularization

Let's modify our training loop to add the L2 penalty to the loss.

In [ ]:
def train_model_l2(model, X_train, y_train, X_test, y_test, epochs=1000, lr=0.01, lambda_l2=0.01):
    """Train model with L2 regularization."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        pred_train = model(X_train).squeeze()
        loss_task = F.mse_loss(pred_train, y_train)
        
        # Add L2 penalty
        l2_penalty = compute_l2_penalty(model)
        loss_total = loss_task + (lambda_l2 / 2) * l2_penalty
        
        loss_total.backward()
        optimizer.step()
        
        # Evaluation (without penalty)
        model.eval()
        with torch.no_grad():
            pred_test = model(X_test).squeeze()
            loss_test = F.mse_loss(pred_test, y_test)
        
        train_losses.append(loss_task.item())  # Track task loss only
        test_losses.append(loss_test.item())
        
        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}: Train Loss = {loss_task:.4f}, Test Loss = {loss_test:.4f}, L2 Penalty = {l2_penalty:.2f}")
    
    return train_losses, test_losses

# Train with L2 regularization
set_seed(42)
model_l2 = OverparameterizedNet().to(device)

train_losses_l2, test_losses_l2 = train_model_l2(
    model_l2, X_train_t, y_train_t, X_test_t, y_test_t, epochs=1000, lambda_l2=0.01
)

Compare regularized vs unregularized training.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(test_losses_no_reg, label='No Regularization', linewidth=2)
plt.plot(test_losses_l2, label='L2 Regularization', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Test Loss')
plt.legend()
plt.title('L2 Regularization Improves Generalization')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
model_l2.eval()
with torch.no_grad():
    X_plot = torch.linspace(0, 2*np.pi, 300).reshape(-1, 1).to(device)
    y_pred_l2 = model_l2(X_plot).squeeze().cpu().numpy()
    y_pred_no_reg = model_no_reg(X_plot).squeeze().cpu().numpy()

plt.scatter(X_train, y_train, alpha=0.6, label='Train Data', s=50, zorder=3)
plt.plot(X_plot.cpu().numpy(), y_pred_no_reg, 'r-', label='No Reg (wiggly)', linewidth=2, alpha=0.7)
plt.plot(X_plot.cpu().numpy(), y_pred_l2, 'b-', label='L2 Reg (smooth)', linewidth=2)
plt.plot(X_plot.cpu().numpy(), np.sin(X_plot.cpu().numpy()), 'g--', label='True Function', linewidth=2)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('L2 Regularization Produces Smoother Fits')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key insight:** L2 regularization produces a **smoother function** that generalizes better. The blue curve is much closer to the true sine wave!

### 3.3 Effect of λ (Lambda)

The strength parameter $\lambda$ controls the trade-off between fitting the data and keeping weights small.

In [ ]:
lambdas = [0.0, 0.001, 0.01, 0.1]
results = {}

for lam in lambdas:
    set_seed(42)
    model = OverparameterizedNet().to(device)
    train_losses, test_losses = train_model_l2(
        model, X_train_t, y_train_t, X_test_t, y_test_t, 
        epochs=1000, lambda_l2=lam
    )
    results[lam] = {
        'model': model,
        'train_losses': train_losses,
        'test_losses': test_losses,
        'final_test_loss': test_losses[-1]
    }

print("\nFinal Test Losses:")
for lam, res in results.items():
    print(f"λ = {lam:6.3f}: {res['final_test_loss']:.4f}")

Visualize how different $\lambda$ values affect the learned function.

In [ ]:
plt.figure(figsize=(14, 4))

X_plot = torch.linspace(0, 2*np.pi, 300).reshape(-1, 1).to(device)

for i, lam in enumerate(lambdas):
    plt.subplot(1, 4, i+1)
    model = results[lam]['model']
    model.eval()
    with torch.no_grad():
        y_pred = model(X_plot).squeeze().cpu().numpy()
    
    plt.scatter(X_train, y_train, alpha=0.6, s=30)
    plt.plot(X_plot.cpu().numpy(), y_pred, 'r-', linewidth=2, label='Prediction')
    plt.plot(X_plot.cpu().numpy(), np.sin(X_plot.cpu().numpy()), 'g--', linewidth=2, label='True')
    plt.title(f'λ = {lam}')
    plt.xlabel('X')
    if i == 0:
        plt.ylabel('y')
        plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observation:** 
- Too small (0.001): Still wiggly, overfits
- Just right (0.01): Smooth and accurate
- Too large (0.1): Underfits, can't capture the sine curve

This is the **regularization strength trade-off**.

## 4. Weight Decay: The Optimizer's View

**Weight decay** is a slightly different way to achieve L2 regularization, implemented directly in the optimizer.

Instead of adding the penalty to the loss, we modify the gradient update:

$$w \leftarrow w - \eta (\nabla \mathcal{L} + \lambda w)$$

This is mathematically equivalent to L2 regularization for standard SGD, but differs slightly for adaptive optimizers like Adam.

### 4.1 Using Weight Decay in PyTorch

PyTorch optimizers have a built-in `weight_decay` parameter.

In [ ]:
def train_model_weight_decay(model, X_train, y_train, X_test, y_test, epochs=1000, lr=0.01, weight_decay=0.01):
    """Train model with weight decay in optimizer."""
    # Weight decay is built into the optimizer!
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_losses = []
    test_losses = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        pred_train = model(X_train).squeeze()
        loss_train = F.mse_loss(pred_train, y_train)
        loss_train.backward()
        optimizer.step()  # Weight decay applied here
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            pred_test = model(X_test).squeeze()
            loss_test = F.mse_loss(pred_test, y_test)
        
        train_losses.append(loss_train.item())
        test_losses.append(loss_test.item())
        
        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}: Train Loss = {loss_train:.4f}, Test Loss = {loss_test:.4f}")
    
    return train_losses, test_losses

# Train with weight decay
set_seed(42)
model_wd = OverparameterizedNet().to(device)

train_losses_wd, test_losses_wd = train_model_weight_decay(
    model_wd, X_train_t, y_train_t, X_test_t, y_test_t, epochs=1000, weight_decay=0.01
)

Compare weight decay vs manual L2 regularization.

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(test_losses_l2, label='Manual L2', linewidth=2)
plt.plot(test_losses_wd, label='Weight Decay', linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Test Loss')
plt.legend()
plt.title('Weight Decay vs Manual L2: Nearly Identical')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final test loss (L2): {test_losses_l2[-1]:.4f}")
print(f"Final test loss (WD): {test_losses_wd[-1]:.4f}")

**Key insight:** For practical purposes, weight decay and L2 regularization produce similar results. **Weight decay is preferred** because it's simpler (no manual loss modification) and better for some optimizers.

## 5. L1 Regularization (Lasso)

**Core idea:** Instead of penalizing $w^2$, penalize $|w|$.

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}} + \lambda \sum_{i} |w_i|$$

**Key difference:** L1 encourages **sparsity** - many weights become exactly zero, effectively doing feature selection.

### 5.1 Why L1 Creates Sparsity

L1's gradient is constant (doesn't shrink near zero), so small weights get pushed all the way to zero.

In [ ]:
# Visualize L1 vs L2 penalties
w = np.linspace(-2, 2, 200)
l1_penalty = np.abs(w)
l2_penalty = w ** 2

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(w, l1_penalty, label='L1: |w|', linewidth=2)
plt.plot(w, l2_penalty, label='L2: w²', linewidth=2)
plt.xlabel('Weight value')
plt.ylabel('Penalty')
plt.legend()
plt.title('L1 vs L2 Penalty')
plt.grid(True, alpha=0.3)
plt.axvline(0, color='black', linewidth=0.5)
plt.axhline(0, color='black', linewidth=0.5)

plt.subplot(1, 2, 2)
l1_grad = np.where(w > 0, 1, -1)
l2_grad = 2 * w
plt.plot(w, l1_grad, label='L1: sign(w)', linewidth=2)
plt.plot(w, l2_grad, label='L2: 2w', linewidth=2)
plt.xlabel('Weight value')
plt.ylabel('Gradient of penalty')
plt.legend()
plt.title('Gradient: L1 Constant, L2 Proportional')
plt.grid(True, alpha=0.3)
plt.axvline(0, color='black', linewidth=0.5)
plt.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

**Key observation:** L1's constant gradient pushes small weights to exactly zero. L2's proportional gradient only shrinks them.

### 5.2 Train with L1 Regularization

In [ ]:
def compute_l1_penalty(model):
    """Compute L1 penalty (sum of absolute weights)."""
    l1_penalty = 0.0
    for param in model.parameters():
        l1_penalty += torch.sum(torch.abs(param))
    return l1_penalty

def train_model_l1(model, X_train, y_train, X_test, y_test, epochs=1000, lr=0.01, lambda_l1=0.001):
    """Train model with L1 regularization."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    sparsity = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        pred_train = model(X_train).squeeze()
        loss_task = F.mse_loss(pred_train, y_train)
        
        # Add L1 penalty
        l1_penalty = compute_l1_penalty(model)
        loss_total = loss_task + lambda_l1 * l1_penalty
        
        loss_total.backward()
        optimizer.step()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            pred_test = model(X_test).squeeze()
            loss_test = F.mse_loss(pred_test, y_test)
            
            # Measure sparsity (% of weights near zero)
            total_params = sum(p.numel() for p in model.parameters())
            near_zero = sum((torch.abs(p) < 0.01).sum().item() for p in model.parameters())
            sparsity_pct = 100 * near_zero / total_params
        
        train_losses.append(loss_task.item())
        test_losses.append(loss_test.item())
        sparsity.append(sparsity_pct)
        
        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}: Train = {loss_task:.4f}, Test = {loss_test:.4f}, Sparsity = {sparsity_pct:.1f}%")
    
    return train_losses, test_losses, sparsity

# Train with L1
set_seed(42)
model_l1 = OverparameterizedNet().to(device)

train_losses_l1, test_losses_l1, sparsity_l1 = train_model_l1(
    model_l1, X_train_t, y_train_t, X_test_t, y_test_t, epochs=1000, lambda_l1=0.001
)

Visualize L1's effect on sparsity over training.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(test_losses_l2, label='L2', linewidth=2)
plt.plot(test_losses_l1, label='L1', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Test Loss')
plt.legend()
plt.title('L1 vs L2: Test Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(sparsity_l1, linewidth=2, color='purple')
plt.xlabel('Epoch')
plt.ylabel('Sparsity (%)')
plt.title('L1 Drives Weights to Zero')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final sparsity: {sparsity_l1[-1]:.1f}% of weights near zero")

### 5.3 Visualize Weight Distribution

In [ ]:
# Collect all weights
weights_l1 = torch.cat([p.flatten() for p in model_l1.parameters()]).detach().cpu().numpy()
weights_l2 = torch.cat([p.flatten() for p in model_l2.parameters()]).detach().cpu().numpy()

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(weights_l2, bins=50, alpha=0.7, label='L2', edgecolor='black')
plt.xlabel('Weight value')
plt.ylabel('Count')
plt.title('L2: Weights Spread Out')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(weights_l1, bins=50, alpha=0.7, label='L1', color='orange', edgecolor='black')
plt.xlabel('Weight value')
plt.ylabel('Count')
plt.title('L1: Many Weights at Zero (Sparse)')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key insight:** L1 creates a **concentrated spike at zero** - true sparsity. L2 just shrinks weights but doesn't eliminate them.

## 6. Dropout: Stochastic Regularization

**Core idea:** During training, randomly set neurons to zero with probability $p$.

**Intuition:** Forces the network to be robust - can't rely on any single neuron. Creates an ensemble effect.

### 6.1 Build Model with Dropout

In [ ]:
class NetWithDropout(nn.Module):
    """Neural network with dropout layers."""
    def __init__(self, hidden_size=100, dropout_p=0.5):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden_size)
        self.dropout1 = nn.Dropout(p=dropout_p)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.dropout2 = nn.Dropout(p=dropout_p)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.dropout3 = nn.Dropout(p=dropout_p)
        self.fc4 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)  # Drop neurons
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = torch.relu(self.fc3(x))
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

model_dropout = NetWithDropout(dropout_p=0.3).to(device)
print(f"Model with dropout created")
print(f"Dropout probability: 0.3 (30% of neurons dropped during training)")

### 6.2 Visualize Dropout in Action

Let's see what dropout does to activations.

In [ ]:
# Create sample input
x_sample = torch.randn(1, 1).to(device)

# Forward pass in training mode (dropout active)
model_dropout.train()
with torch.no_grad():
    activations_train = []
    for _ in range(5):
        x = torch.relu(model_dropout.fc1(x_sample))
        x_dropped = model_dropout.dropout1(x)
        activations_train.append(x_dropped.cpu().numpy().flatten())

# Forward pass in eval mode (dropout inactive)
model_dropout.eval()
with torch.no_grad():
    x = torch.relu(model_dropout.fc1(x_sample))
    x_eval = model_dropout.dropout1(x)
    activations_eval = x_eval.cpu().numpy().flatten()

# Visualize
plt.figure(figsize=(12, 4))

for i in range(5):
    plt.subplot(1, 6, i+1)
    plt.bar(range(20), activations_train[i][:20])
    plt.title(f'Train {i+1}')
    plt.ylim([0, max(activations_eval[:20]) * 1.1])
    if i == 0:
        plt.ylabel('Activation')

plt.subplot(1, 6, 6)
plt.bar(range(20), activations_eval[:20], color='green')
plt.title('Eval')
plt.ylim([0, max(activations_eval[:20]) * 1.1])

plt.tight_layout()
plt.show()

print("Notice: Training mode randomly zeros out neurons (different each time)")
print("Eval mode keeps all neurons active")

### 6.3 Train with Dropout

In [ ]:
# Train with dropout
set_seed(42)
model_dropout = NetWithDropout(dropout_p=0.3).to(device)

train_losses_dropout, test_losses_dropout = train_model(
    model_dropout, X_train_t, y_train_t, X_test_t, y_test_t, epochs=1000
)

Compare dropout to other regularization methods.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(test_losses_no_reg, label='No Regularization', linewidth=2)
plt.plot(test_losses_l2, label='L2', linewidth=2)
plt.plot(test_losses_dropout, label='Dropout', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Test Loss')
plt.legend()
plt.title('Dropout vs L2 Regularization')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
model_dropout.eval()
with torch.no_grad():
    X_plot = torch.linspace(0, 2*np.pi, 300).reshape(-1, 1).to(device)
    y_pred_dropout = model_dropout(X_plot).squeeze().cpu().numpy()
    y_pred_l2 = model_l2(X_plot).squeeze().cpu().numpy()

plt.scatter(X_train, y_train, alpha=0.6, s=50, zorder=3)
plt.plot(X_plot.cpu().numpy(), y_pred_l2, 'b-', label='L2', linewidth=2, alpha=0.7)
plt.plot(X_plot.cpu().numpy(), y_pred_dropout, 'purple', label='Dropout', linewidth=2)
plt.plot(X_plot.cpu().numpy(), np.sin(X_plot.cpu().numpy()), 'g--', label='True', linewidth=2)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Dropout Produces Smooth Predictions')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key insight:** Dropout is particularly effective for **deep networks** where co-adaptation between neurons is a problem. It's complementary to L2 - you can use both!

### 6.4 Effect of Dropout Probability

In [ ]:
dropout_probs = [0.0, 0.2, 0.5, 0.7]
dropout_results = {}

for p in dropout_probs:
    set_seed(42)
    model = NetWithDropout(dropout_p=p).to(device)
    train_losses, test_losses = train_model(
        model, X_train_t, y_train_t, X_test_t, y_test_t, epochs=1000
    )
    dropout_results[p] = {
        'model': model,
        'test_losses': test_losses,
        'final_test_loss': test_losses[-1]
    }

print("\nFinal Test Losses:")
for p, res in dropout_results.items():
    print(f"Dropout p = {p:.1f}: {res['final_test_loss']:.4f}")

Visualize the effect of different dropout probabilities.

In [ ]:
plt.figure(figsize=(10, 4))

for p, res in dropout_results.items():
    plt.plot(res['test_losses'], label=f'p={p}', linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Test Loss')
plt.legend()
plt.title('Effect of Dropout Probability')
plt.grid(True, alpha=0.3)
plt.show()

**Key observation:**
- p=0.0: No regularization, overfits
- p=0.2-0.5: Good balance
- p=0.7: Too much dropout, underfits (can't learn)

**Rule of thumb:** 0.3-0.5 works well for most cases.

## 7. Early Stopping: The Simplest Regularization

**Core idea:** Stop training when validation loss starts increasing.

**Intuition:** Before overfitting begins, the model has learned the signal but not the noise. Stop there!

### 7.1 Implement Early Stopping

In [ ]:
def train_with_early_stopping(model, X_train, y_train, X_test, y_test, 
                               epochs=1000, lr=0.01, patience=50):
    """Train with early stopping."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    
    best_test_loss = float('inf')
    best_epoch = 0
    best_model_state = None
    epochs_without_improvement = 0
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        pred_train = model(X_train).squeeze()
        loss_train = F.mse_loss(pred_train, y_train)
        loss_train.backward()
        optimizer.step()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            pred_test = model(X_test).squeeze()
            loss_test = F.mse_loss(pred_test, y_test)
        
        train_losses.append(loss_train.item())
        test_losses.append(loss_test.item())
        
        # Early stopping logic
        if loss_test < best_test_loss:
            best_test_loss = loss_test
            best_epoch = epoch
            best_model_state = model.state_dict().copy()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            print(f"Best test loss was {best_test_loss:.4f} at epoch {best_epoch+1}")
            # Restore best model
            model.load_state_dict(best_model_state)
            break
        
        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}: Train = {loss_train:.4f}, Test = {loss_test:.4f}")
    
    return train_losses, test_losses, best_epoch

# Train with early stopping
set_seed(42)
model_early = OverparameterizedNet().to(device)

train_losses_early, test_losses_early, best_epoch = train_with_early_stopping(
    model_early, X_train_t, y_train_t, X_test_t, y_test_t, epochs=1000, patience=50
)

Visualize when early stopping would kick in.

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(train_losses_no_reg, label='Train Loss (no early stop)', linewidth=2, alpha=0.6)
plt.plot(test_losses_no_reg, label='Test Loss (no early stop)', linewidth=2, alpha=0.6)
plt.plot(train_losses_early, label='Train Loss (early stop)', linewidth=2)
plt.plot(test_losses_early, label='Test Loss (early stop)', linewidth=2)
plt.axvline(best_epoch, color='red', linestyle='--', linewidth=2, label=f'Best Epoch ({best_epoch+1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Early Stopping: Stop Before Overfitting')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nWithout early stopping - Final test loss: {test_losses_no_reg[-1]:.4f}")
print(f"With early stopping - Best test loss: {test_losses_early[best_epoch]:.4f}")
print(f"Improvement: {(test_losses_no_reg[-1] - test_losses_early[best_epoch]) / test_losses_no_reg[-1] * 100:.1f}%")

**Key insight:** Early stopping is **implicit regularization** - we don't change the model or loss, just when we stop training. It's simple and effective!

## 8. Combining Regularization Techniques

In practice, we often combine multiple regularization methods for best results.

### 8.1 L2 + Dropout + Early Stopping

In [ ]:
def train_combined(model, X_train, y_train, X_test, y_test, 
                   epochs=1000, lr=0.01, weight_decay=0.01, patience=50):
    """Train with weight decay + dropout + early stopping."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_losses = []
    test_losses = []
    
    best_test_loss = float('inf')
    best_epoch = 0
    best_model_state = None
    epochs_without_improvement = 0
    
    for epoch in range(epochs):
        # Training (dropout active in .train() mode)
        model.train()
        optimizer.zero_grad()
        pred_train = model(X_train).squeeze()
        loss_train = F.mse_loss(pred_train, y_train)
        loss_train.backward()
        optimizer.step()
        
        # Evaluation (dropout inactive in .eval() mode)
        model.eval()
        with torch.no_grad():
            pred_test = model(X_test).squeeze()
            loss_test = F.mse_loss(pred_test, y_test)
        
        train_losses.append(loss_train.item())
        test_losses.append(loss_test.item())
        
        # Early stopping
        if loss_test < best_test_loss:
            best_test_loss = loss_test
            best_epoch = epoch
            best_model_state = model.state_dict().copy()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            model.load_state_dict(best_model_state)
            break
        
        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}: Train = {loss_train:.4f}, Test = {loss_test:.4f}")
    
    return train_losses, test_losses, best_epoch

# Train with combined regularization
set_seed(42)
model_combined = NetWithDropout(dropout_p=0.3).to(device)

train_losses_combined, test_losses_combined, best_epoch_combined = train_combined(
    model_combined, X_train_t, y_train_t, X_test_t, y_test_t, 
    epochs=1000, weight_decay=0.01, patience=50
)

### 8.2 Compare All Methods

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(test_losses_no_reg, label='No Regularization', linewidth=2, alpha=0.7)
plt.plot(test_losses_l2, label='L2 Only', linewidth=2, alpha=0.7)
plt.plot(test_losses_dropout, label='Dropout Only', linewidth=2, alpha=0.7)
plt.plot(test_losses_early, label='Early Stopping Only', linewidth=2, alpha=0.7)
plt.plot(test_losses_combined, label='L2 + Dropout + Early Stop', linewidth=3)
plt.xlabel('Epoch')
plt.ylabel('Test Loss')
plt.legend()
plt.title('Comparing Regularization Techniques')
plt.grid(True, alpha=0.3)
plt.ylim([0, max(test_losses_no_reg[:100]) * 0.5])

plt.subplot(1, 2, 2)
methods = ['None', 'L2', 'Dropout', 'Early\nStop', 'Combined']
final_losses = [
    test_losses_no_reg[-1],
    test_losses_l2[-1],
    test_losses_dropout[-1],
    test_losses_early[best_epoch],
    test_losses_combined[best_epoch_combined]
]
colors = ['red', 'blue', 'purple', 'orange', 'green']
plt.bar(methods, final_losses, color=colors, alpha=0.7, edgecolor='black')
plt.ylabel('Final Test Loss')
plt.title('Final Performance Comparison')
plt.grid(True, alpha=0.3, axis='y')

# Add values on bars
for i, v in enumerate(final_losses):
    plt.text(i, v + 0.005, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nFinal Test Losses:")
for method, loss in zip(methods, final_losses):
    print(f"{method:12s}: {loss:.4f}")

### 8.3 Visual Comparison of Predictions

In [ ]:
plt.figure(figsize=(15, 4))

models = [model_no_reg, model_l2, model_dropout, model_early, model_combined]
titles = ['No Reg', 'L2', 'Dropout', 'Early Stop', 'Combined']

X_plot = torch.linspace(0, 2*np.pi, 300).reshape(-1, 1).to(device)

for i, (model, title) in enumerate(zip(models, titles)):
    plt.subplot(1, 5, i+1)
    model.eval()
    with torch.no_grad():
        y_pred = model(X_plot).squeeze().cpu().numpy()
    
    plt.scatter(X_train, y_train, alpha=0.6, s=30)
    plt.plot(X_plot.cpu().numpy(), y_pred, 'r-', linewidth=2, label='Pred')
    plt.plot(X_plot.cpu().numpy(), np.sin(X_plot.cpu().numpy()), 'g--', linewidth=2, label='True')
    plt.title(title)
    plt.xlabel('X')
    if i == 0:
        plt.ylabel('y')
        plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim([-1.5, 1.5])

plt.tight_layout()
plt.show()

**Key observation:** Combined regularization produces the **smoothest, most accurate fit**. Each technique addresses different aspects of overfitting!

## 9. Key Takeaways

### Regularization Techniques Summary

| Technique | How it works | Best for | Typical values |
|-----------|-------------|----------|----------------|
| **L2 (Ridge)** | Penalizes $w^2$ | General purpose | λ = 0.001-0.1 |
| **Weight Decay** | L2 in optimizer | Same as L2 (easier) | 0.001-0.1 |
| **L1 (Lasso)** | Penalizes $|w|$ | Feature selection | λ = 0.0001-0.01 |
| **Dropout** | Randomly drop neurons | Deep networks | p = 0.3-0.5 |
| **Early Stopping** | Stop when val loss increases | Always! | patience = 10-50 |

### Core Intuitions

1. **Overfitting = memorization**: Models with too much capacity fit noise instead of signal

2. **Regularization = constraints**: Adding constraints forces simpler, more generalizable solutions

3. **L2 smooths**: Penalizes large weights → smoother functions

4. **L1 sparsifies**: Pushes weights to exactly zero → automatic feature selection

5. **Dropout = ensemble**: Training many sub-networks → robust predictions

6. **Early stopping = free lunch**: No hyperparameters, always beneficial

### Practical Guidelines

**Start with:**
- Weight decay (0.01) + Early stopping (patience=10-20)

**If still overfitting:**
- Add dropout (0.3-0.5) to hidden layers
- Increase weight decay

**If you need sparsity:**
- Use L1 regularization

**Remember:**
- More data is better than more regularization
- Combine techniques for best results
- Tune on validation set, evaluate on test set

## 10. Further Exploration

Try modifying the hyperparameters and observe the effects:

```python
# Experiment 1: Very strong L2
model = OverparameterizedNet().to(device)
train_model_l2(model, X_train_t, y_train_t, X_test_t, y_test_t, lambda_l2=1.0)

# Experiment 2: Very high dropout
model = NetWithDropout(dropout_p=0.9).to(device)
train_model(model, X_train_t, y_train_t, X_test_t, y_test_t)

# Experiment 3: No early stopping patience
model = OverparameterizedNet().to(device)
train_with_early_stopping(model, X_train_t, y_train_t, X_test_t, y_test_t, patience=5)
```

**Questions to explore:**
- What happens with more training data?
- How do regularization needs change with model size?
- Can you overregularize?